In [1]:
# import pandas as pd
#
# # Đọc file JSON dạng array
# df = pd.read_json("data/law_content_chunks.json")
#
# # Ghi ra file Parquet
# df.to_parquet("data/law_html_content.parquet",
#     index=False,
#     compression="zstd",
#     engine="pyarrow"
# )

In [1]:
import json

import pandas as pd

df_raw = pd.read_parquet("data/law_html_content.parquet", engine="pyarrow")

df_raw.head(3)

,law_id,url,content_html,chunks,error
0,14/2022/TT-NHNN,https://thuvienphapluat.vn/van-ban/Tien-te-Nga...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[{'content': 'NGÂN HÀNG NHÀ NƯỚC VIỆT NAM -...,None
1,11/2022/TT-NHNN,https://thuvienphapluat.vn/van-ban/Tien-te-Nga...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[{'content': 'NGÂN HÀNG NHÀ NƯỚC VIỆT NAM -...,None
2,02/2007/TT-BNV,https://thuvienphapluat.vn/van-ban/Lao-dong-Ti...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[{'content': 'BỘ NỘI VỤ ****** CỘNG HOÀ ...,None


In [2]:
import numpy as np

df = df_raw.copy()
df = df[df['chunks'].apply(lambda x: len(x) == 0)]
print(len(df))
df.head(3)

171


,law_id,url,content_html,chunks,error
3,17/2019/TT-NHNN,https://thuvienphapluat.vn/van-ban/Tien-te-Nga...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[],None
70,123/2020/NĐ-CP,https://thuvienphapluat.vn/van-ban/Ke-toan-Kie...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[],None
139,04/2020/TT-BTP,https://thuvienphapluat.vn/van-ban/Quyen-dan-s...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[],None


In [4]:
# df_raw['chunks'] = df_raw['chunks'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else x)
# chunks = df_raw["chunks"][0]
# for chunk in chunks:
#     print(" - ".join(chunk["titles"]))
#     print(chunk["content"])
#     print("-"*50)

In [5]:
from bs4 import BeautifulSoup

def get_deepest(div, depth=0, max_depth=2):
    # Nếu đạt tới độ sâu tối đa thì dừng
    if depth >= max_depth:
        return div

    # Tìm các div con trực tiếp
    child_divs = div.find_all('div', recursive=False)
    if not child_divs:
        return div

    # Đệ quy vào div con đầu tiên
    return get_deepest(child_divs[0], depth + 1, max_depth)

def chunk_document(html):
    soup = BeautifulSoup(html, 'html.parser')
    container = soup.find('div', class_='content1')
    if not container:
        return []

    container = get_deepest(container)

    # Định nghĩa thứ tự các cấp tiêu đề
    levels   = ['loai', 'chuong', 'muc', 'dieu']
    current  = { lvl: None for lvl in levels }
    chunks   = []
    buffer   = []

    def close_chunk():
        # Chỉ đóng chunk khi đã gặp loai_… (seen_loai) và buffer không rỗng
        if buffer:
            titles = [current[l] for l in levels if current[l]]
            chunks.append({
                'titles': titles,
                'content': '\n'.join(buffer).strip()
            })
        buffer.clear()

    for tag in container.find_all(recursive=False):
        check_flag = False
        first_a = tag.find('a')
        if first_a and first_a.has_attr('name'):
            # Nếu thẻ <a> có thuộc tính name, thì đó là tiêu đề
            name = first_a['name']
            for lvl in levels:
                if name.startswith(lvl):
                    check_flag = True
                    if name.endswith("_name"):
                        if current[lvl] is None:
                            current[lvl] = first_a.get_text(strip=True)
                        else:
                            current[lvl] += " " + first_a.get_text(strip=True)
                    else:
                        close_chunk()
                        idx = levels.index(lvl)
                        # reset các cấp thấp hơn
                        for lower in levels[idx+1:]:
                            current[lower] = None
                        # lấy text làm title
                        current[lvl]  = first_a.get_text(strip=True)
                        break
            if not check_flag:
                text = tag.get_text(separator=' ', strip=True)
                if text:
                    buffer.append(text)
        else:
            text = tag.get_text(separator=' ', strip=True)
            if text:
                buffer.append(text)
    close_chunk()
    return chunks

html_content = df_raw["content_html"][9].strip()
# html_content = df_raw["content_html"][0].strip()
chunks = chunk_document(html_content)
print(chunks)

[{'titles': [], 'content': 'QUỐC HỘI ------- CỘNG HÒA XÃ HỘI CHỦ\r\n  NGHĨA VIỆT NAM Độc lập – Tự do – Hạnh phúc --------- Luật số:\r\n  47/2010/QH12 Hà Nội, ngày 16\r\n  tháng 6 năm 2010'}, {'titles': ['LUẬT CÁC TỔ CHỨC\r\nTÍN DỤNG'], 'content': 'Căn cứ Hiến\r\npháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 1992 đã được sửa đổi, bổ\r\nsung một số điều theo Nghị quyết số 51/2001/QH10 ;\nQuốc hội ban hành Luật các tổ chức tín dụng.'}, {'titles': ['LUẬT CÁC TỔ CHỨC\r\nTÍN DỤNG', 'Chương I NHỮNG\r\nQUY ĐỊNH CHUNG', 'Điều 1. Phạm vi điều\r\nchỉnh'], 'content': 'Luật này quy định về việc thành lập, tổ chức,\r\nhoạt động, kiểm soát đặc biệt, tổ chức lại, giải thể tổ chức tín dụng; việc\r\nthành lập, tổ chức, hoạt động của chi nhánh ngân hàng nước ngoài, văn phòng đại\r\ndiện của tổ chức tín dụng nước ngoài, tổ chức nước ngoài khác có hoạt động ngân\r\nhàng.'}, {'titles': ['LUẬT CÁC TỔ CHỨC\r\nTÍN DỤNG', 'Chương I NHỮNG\r\nQUY ĐỊNH CHUNG', 'Điều 2. Đối tượng áp\r\ndụng'], 'content': 'Luật n

In [6]:
df = df_raw.copy()
for i in range(len(df)):
    # if len(df["chunks"][i]) != 0:
    #     continue
    print(f"{i}/{len(df)}")
    chunks = chunk_document(df["content_html"][i].strip())
    df.at[i, "chunks"] = chunks
df[df['chunks'].apply(lambda x: len(x) == 0)]

0/2156
1/2156
2/2156
3/2156
4/2156
5/2156
6/2156
7/2156
8/2156
9/2156
10/2156
11/2156
12/2156
13/2156
14/2156
15/2156
16/2156
17/2156
18/2156
19/2156
20/2156
21/2156
22/2156
23/2156
24/2156
25/2156
26/2156
27/2156
28/2156
29/2156
30/2156
31/2156
32/2156
33/2156
34/2156
35/2156
36/2156
37/2156
38/2156
39/2156
40/2156
41/2156
42/2156
43/2156
44/2156
45/2156
46/2156
47/2156
48/2156
49/2156
50/2156
51/2156
52/2156
53/2156
54/2156
55/2156
56/2156
57/2156
58/2156
59/2156
60/2156
61/2156
62/2156
63/2156
64/2156
65/2156
66/2156
67/2156
68/2156
69/2156
70/2156
71/2156
72/2156
73/2156
74/2156
75/2156
76/2156
77/2156
78/2156
79/2156
80/2156
81/2156
82/2156
83/2156
84/2156
85/2156
86/2156
87/2156
88/2156
89/2156
90/2156
91/2156
92/2156
93/2156
94/2156
95/2156
96/2156
97/2156
98/2156
99/2156
100/2156
101/2156
102/2156
103/2156
104/2156
105/2156
106/2156
107/2156
108/2156
109/2156
110/2156
111/2156
112/2156
113/2156
114/2156
115/2156
116/2156
117/2156
118/2156
119/2156
120/2156
121/2156
122/2156
123

,law_id,url,content_html,chunks,error
83,02/2020/TT-NHNN,https://thuvienphapluat.vn/van-ban/Thuong-mai/...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[],None
104,130/2003/QĐ-TTg,https://thuvienphapluat.vn/van-ban/Tien-te-Nga...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[],None
507,03/2013/TTLT-TANDTC-VKSNDTC,https://thuvienphapluat.vn/van-ban/Thu-tuc-To-...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[],None
521,590/QĐ-VKSTC-V3,https://thuvienphapluat.vn/van-ban/Thu-tuc-To-...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[],None
523,386/2016/QĐ-TANDTC,https://thuvienphapluat.vn/van-ban/Thu-tuc-To-...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[],None
714,46/2020/TT-BGDĐT,https://thuvienphapluat.vn/van-ban/Giao-duc/Th...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[],None
842,1919/QĐ-BTC,https://thuvienphapluat.vn/van-ban/Xuat-nhap-k...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[],None
860,188/QĐ-TCHQ,https://thuvienphapluat.vn/van-ban/Xuat-nhap-k...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[],None
1069,70/2018/QH14,https://thuvienphapluat.vn/van-ban/Tai-chinh-n...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[],None
1379,963/QĐ-BCT,https://thuvienphapluat.vn/van-ban/Thuong-mai/...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[],None


In [7]:
df.to_parquet("../data/law_content_chunks.parquet",
    index=False,
    compression="zstd",
    engine="pyarrow"
)